[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc3_stats/corrections/seance3_correction.ipynb)

# Séance 3.3 — Relier deux variables — y a-t-il un lien ?

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- découper une variable continue en catégories avec `pd.cut`
- tester le lien entre deux variables qualitatives avec un khi-deux
- lire un tableau d'effectifs attendus pour dire *où* est la dépendance
- mesurer un lien entre deux variables quantitatives (Pearson, Spearman)
- reconnaître les trois pièges de la corrélation : extrêmes, non-linéarité, causalité

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc3_stats/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
cmd = pd.read_csv(BASE + "commandes.csv")

cmd["taille"] = pd.cut(cmd["ca"], [0, 200, 500, 1e9],
                       labels=["petite", "moyenne", "grande"])
print(cmd["taille"].value_counts())

### Exercice 1 — Découper les montants

> **Votre mission :**
> - La colonne `taille` a été créée dans la cellule de préparation (`petite`, `moyenne`, `grande`).
> - Combien y a-t-il de grosses commandes ? → `nb_grande`

In [ ]:
# (colonne == valeur) donne des True/False ; sum() les compte
nb_grande = (cmd["taille"] == "grande").sum()

print(nb_grande, "grosses commandes")

In [ ]:
verifier("1 - nombre de grosses commandes", nb_grande == 685,
         "les etiquettes sont petite, moyenne et grande")

### Exercice 2 — Le tableau croisé

> **Votre mission :**
> - Croiser `pays` (en lignes) et `taille` (en colonnes) pour les quatre pays les plus présents → `tab`.
> - Combien de grosses commandes irlandaises ? → `irl_grande`

In [ ]:
top4 = cmd["pays"].value_counts().head(4).index
sub = cmd.query("pays in @top4")

# index = lignes, colonnes = ce qu'on croise avec
tab = pd.crosstab(sub["pays"], sub["taille"])
irl_grande = tab.loc["Irlande", "grande"]
print(tab)

In [ ]:
verifier("2 - grosses commandes irlandaises", irl_grande == 157,
         "pd.crosstab(lignes, colonnes) puis .loc['Irlande', 'grande']")

### Exercice 3 — Le khi-deux

> **Votre mission :**
> - Tester le lien entre pays et taille de commande sur `tab`.
> - Récupérer la p-value → `p_khi2`, et conclure au seuil de 5 % → `dependant` (`True`/`False`).

In [ ]:
# chi2_contingency renvoie quatre choses : la statistique, la p-value,
# les degres de liberte et le tableau des effectifs attendus
khi2, p_khi2, ddl, attendus = stats.chi2_contingency(tab)

dependant = p_khi2 < 0.05
print("p =", p_khi2, "| dependance :", dependant)

In [ ]:
verifier("3 - dependance pays / taille", bool(dependant) and p_khi2 < 1e-20,
         "passez le tableau croise a stats.chi2_contingency")

### Exercice 4 — Où est la dépendance ?

> **Votre mission :**
> - Construire le tableau des écarts (observé − attendu) → `ecarts`.
> - Quel est l'écart des grosses commandes britanniques ? → `ecart_uk` (arrondi à 1 décimale)

In [ ]:
att = pd.DataFrame(attendus, index=tab.index, columns=tab.columns)
ecarts = tab - att

# Negatif : il y a MOINS de grosses commandes britanniques que ce qu'on
# attendrait si le pays n'y etait pour rien
ecart_uk = round(ecarts.loc["Royaume-Uni", "grande"], 1)
print(ecarts.round(1))

In [ ]:
verifier("4 - ecart des grosses commandes britanniques", ecart_uk == -82.7,
         "observe moins attendu, donc tab - att")

### Exercice 5 — Un lien qui n'existe pas

> **Votre mission :**
> - Le jour de la semaine influence-t-il la taille des commandes ?
> - Croiser `jour` et `taille`, tester, et mettre la p-value dans `p_jour` (arrondie à 4 décimales).

In [ ]:
tab_jour = pd.crosstab(cmd["jour"], cmd["taille"])

# chi2_contingency renvoie un quadruplet : la p-value est en position 1
p_jour = round(stats.chi2_contingency(tab_jour)[1], 4)

print("p =", p_jour)

# 0,7866 : la taille des commandes ne depend pas du jour. Une bonne
# nouvelle a ecrire dans une note, au meme titre qu'une dependance.

In [ ]:
verifier("5 - jour et taille de commande", p_jour == 0.7866,
         "la p-value est le deuxieme element renvoye, donc l'indice 1")

### Exercice 6 — Les corrélations d'un coup

> **Votre mission :**
> - Afficher la matrice de corrélation de `ca`, `nart` et `qte`.
> - En extraire la corrélation entre `ca` et `qte` → `r_ca_qte` (arrondie à 3 décimales).

In [ ]:
print(cmd[["ca", "nart", "qte"]].corr().round(3))

r_ca_qte = round(cmd["ca"].corr(cmd["qte"]), 3)
print(r_ca_qte)

In [ ]:
verifier("6 - correlation ca / qte", r_ca_qte == 0.848,
         "df['ca'].corr(df['qte'])")

### Exercice 7 — Pearson contre Spearman

> **Votre mission :**
> - Calculer les deux corrélations entre `ca` et `nart` → `r_pearson` et `r_spear` (arrondies à 3 décimales).
> - Laquelle est la plus élevée, et pourquoi ?

In [ ]:
r_pearson = round(cmd["ca"].corr(cmd["nart"]), 3)

# Spearman raisonne sur les RANGS : il ne demande pas que la relation
# soit droite, seulement qu'elle monte
r_spear = round(cmd["ca"].corr(cmd["nart"], method="spearman"), 3)

print("Pearson", r_pearson, "| Spearman", r_spear)

In [ ]:
verifier("7a - Pearson", r_pearson == 0.382, "c'est la methode par defaut")
verifier("7b - Spearman", r_spear == 0.671, "method='spearman'")

### Exercice 8 — Le poids de 1 % des lignes

> **Votre mission :**
> - Retirer les 1 % de commandes les plus grosses → `sans`.
> - Recalculer la corrélation entre `ca` et `nart` sur ce sous-ensemble → `r_sans` (arrondie à 3 décimales).

In [ ]:
# quantile(0.99) : le montant au-dessus duquel se trouvent les 1 % du haut
seuil = cmd["ca"].quantile(0.99)
sans = cmd.query("ca < @seuil")

r_sans = round(sans["ca"].corr(sans["nart"]), 3)
print(len(cmd) - len(sans), "lignes retirees | correlation :", r_sans)

# 0,382 -> 0,489 en retirant 20 lignes sur 1 955.

In [ ]:
verifier("8 - correlation sans les extremes", r_sans == 0.489,
         "les 1 % du haut commencent au quantile 0.99")

### Exercice 9 — Le même chiffre, deux pays

> **Votre mission :**
> - Calculer la corrélation `ca` / `nart` **en Belgique** → `r_be`, puis **en Irlande** → `r_irl`.
> - Arrondir à 3 décimales. Comparez au 0,382 global.

In [ ]:
be = cmd.query("pays == 'Belgique'")
irl = cmd.query("pays == 'Irlande'")

r_be = round(be["ca"].corr(be["nart"]), 3)
r_irl = round(irl["ca"].corr(irl["nart"]), 3)
print("Belgique", r_be, "| Irlande", r_irl)

# 0,903 contre 0,234, pour un 0,382 global qui n'est le chiffre de
# personne. Un coefficient calcule sur un melange de populations
# differentes ne decrit aucune d'entre elles.

In [ ]:
verifier("9a - correlation belge", r_be == 0.903, "filtrez d'abord, correlez ensuite")
verifier("9b - correlation irlandaise", r_irl == 0.234,
         "le nom du pays s'ecrit Irlande, avec une majuscule")

### Exercice 10 — Question de synthèse

> **Votre mission :**
> - On vous demande : *« le nombre de références commandées explique-t-il le montant ? »*
> - Calculer la corrélation de Spearman `ca` / `nart` sur le seul Royaume-Uni → `r_uk` (3 décimales).
> - Puis tracer le nuage de points correspondant.
> - Enfin, écrivez votre réponse en commentaire — en trois phrases maximum.

In [ ]:
uk = cmd.query("pays == 'Royaume-Uni'")
r_uk = round(uk["ca"].corr(uk["nart"], method="spearman"), 3)

uk.plot(kind="scatter", x="nart", y="ca", alpha=0.3, figsize=(7, 4))
plt.show()
print("Spearman au Royaume-Uni :", r_uk)

# Une reponse possible :
# "Oui, le lien est net sur le marche britannique (Spearman 0,57) : les
#  commandes qui portent sur plus de references sont plus cheres. Mais le
#  coefficient global (0,38) est trompeur, parce qu'il melange des marches
#  de detail et deux grossistes irlandais. Il faut raisonner marche par
#  marche, pas sur le fichier entier."

In [ ]:
verifier("10 - Spearman au Royaume-Uni", r_uk == 0.574,
         "filtrez sur le Royaume-Uni, puis method='spearman'")